<p style="text-align:center">
    <a href="https://tukkalearn.vercel.app" target="_blank">
    <img src="https://raw.githubusercontent.com/itzDM/publicAssets/refs/heads/main/opengraph-image.png" width="250"  alt="Tukka Learn">
    </a>
</p>


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv('https://raw.githubusercontent.com/tukkaLearn/datasets/refs/heads/main/orders_full_nov2025.csv')
df.head()

### Convert string to proper datetime


In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce', utc=True)

df["delivery_date"] = pd.to_datetime(df["delivery_date"], errors='coerce', utc=True)

print(f"Failed Order dates → {df['order_date'].isna().sum()} (now NaT)\nFailed Delivery dates → {df['delivery_date'].isna().sum()} (now NaT)")
df[['order_id', 'order_date','delivery_date']].head(10)

### Extract everything with .dt


In [ ]:
df['date_only']      = df['order_date'].dt.date
df['hour']           = df['order_date'].dt.hour
df['day_name']       = df['order_date'].dt.day_name()
df['is_weekend']     = df['order_date'].dt.weekday >= 5
df['week_number']    = df['order_date'].dt.weekday        # 0=Mon
df['month_end']      = df['order_date'].dt.is_month_end

df[['order_id', 'order_date', 'date_only', 'hour', 'day_name', 'is_weekend', 'week_number', 'month_end']].head()

### Time differences – customer journey


In [ ]:
df["delivery_time_minutes"]=df["delivery_date"]-df["order_date"]
df["delivery_time_minutes"]=df["delivery_time_minutes"].dt.total_seconds()/60

In [ ]:
df["delivery_time_minutes"].mean()

In [ ]:
display(df["delivery_time_minutes"].quantile(0.95))

### Time zones – global business reality


In [ ]:
df['created_pacific'] = df['order_date'].dt.tz_convert('US/Pacific')
df['created_india'] = df['order_date'].dt.tz_convert('Asia/Kolkata')

display(df['created_pacific'].dt.hour.value_counts().idxmax())
display(df['created_india'].dt.hour.value_counts().idxmax())

### Filter by date range (Black Friday week)


In [ ]:
df = df.set_index('order_date').sort_index()


bf_start = '2025-11-28'
bf_end   = '2025-12-01'

bf_df = df.loc[bf_start:bf_end]
bf_df

### Generate complete date range (find missing hours)


In [ ]:
df.reset_index(inplace=True)
# Simulate hourly data
hourly = df.set_index('order_date').resample('h')['total_amount'].sum().reset_index()

# Create perfect hourly index for November
full_hours = pd.date_range('2025-11-01', '2025-11-30 23:59:59', freq='h')

hourly_full = hourly.set_index('order_date').reindex(full_hours).fillna(0).reset_index()
hourly_full = hourly_full.rename(columns={'index': 'hour'})

print(f"Missing hours before reindex: {hourly.shape[0]}")
print(f"After perfect grid: {hourly_full.shape[0]} hours → {hourly_full['total_amount'].eq(0).sum()} quiet hours")
hourly_full.tail()

### Resample() – from hourly to daily/weekly


In [ ]:
daily  = df.set_index('order_date').resample('D')['total_amount'].sum()
weekly = df.set_index('order_date').resample('W-MON')['total_amount'].sum()
monthly = df.set_index('order_date').resample('ME')['total_amount'].sum()

print(f"Daily Revenue: ${daily.mean():,.0f} ± ${daily.std():,.0f}")
print(f"Weekly Revenue: ${weekly.mean():,.0f} ± ${weekly.std():,.0f}")
print(f"Monthly Revenue: ${monthly.mean():,.0f} ± ${monthly.std():,.0f}")



### Shifting – compare with yesterday


In [ ]:
df['revenue_yesterday'] = df['revenue'].shift(1)
df['growth_vs_yesterday'] = (
    df['revenue'] / df['revenue_yesterday'] - 1
)

df.sort_values('growth_vs_yesterday', ascending=False).head(5)


### Percentage change – stock-style metrics


In [ ]:
hourly_revenue = df.groupby(df['order_date'].dt.hour)['total_amount'].mean()
peak_hour = hourly_revenue.idxmax()
print(f"The peak hour is {peak_hour}:00")

### Rolling windows – 7-day moving average


In [ ]:
df_daily = df.set_index('order_date').sort_index()
df_daily['revenue_7d'] = df_daily['total_amount'].rolling('7D').sum()
df_daily['revenue_7d_ma'] = df_daily['total_amount'].rolling(window=7).mean()

df_daily[['total_amount', 'revenue_7d_ma']]

<hr>
<div style="text-align:center">
  <h3 style="color:orange">|| राम नाम सत्य है ||</h3>
  <h4>Authour : सीता राम जी </h4>
   <h5 style="color:skyblue"><i>© All Rights Reserved</i></h5>
</div>
